In [ ]:
# 1. Cài đặt thư viện cần thiết
!pip install -q datasets scikit-learn pandas numpy

import os
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import joblib

# 2. Cấu hình đường dẫn lưu trữ
from google.colab import drive
drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/MFND_Baseline_TFIDF_SVM"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 3. Tải dữ liệu MiRAGeNews
print("⏳ Loading dataset...")
dataset = load_dataset("anson-huang/mirage-news")

# Chuẩn bị dữ liệu Train/Val
X_train = dataset["train"]["text"]
y_train = dataset["train"]["label"]
X_val = dataset["validation"]["text"]
y_val = dataset["validation"]["label"]

# 4. Trích xuất đặc trưng TF-IDF
# Sử dụng n-gram (1,2) để bắt được các cụm từ đặc trưng
print("📊 Vectorizing text with TF-IDF...")
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

# 5. Huấn luyện Linear SVM
# Dùng class_weight='balanced' vì dữ liệu tin giả thường mất cân bằng
print("🚀 Training Linear SVM...")
svm = LinearSVC(C=1.0, class_weight="balanced", max_iter=10000, random_state=42)
svm.fit(X_train_tfidf, y_train)

# 6. Đánh giá Đa mục tiêu (In-domain & Out-domain)
test_splits = ["test1_nyt_mj", "test2_bbc_dalle", "test3_cnn_dalle", "test4_bbc_sdxl", "test5_cnn_sdxl"]
results_storage = {}

print("\n🔍 ĐÁNH GIÁ TRÊN CÁC TẬP TEST:")
for split in test_splits:
    X_test = dataset[split]["text"]
    y_test = dataset[split]["label"]

    X_test_tfidf = tfidf.transform(X_test)
    y_pred = svm.predict(X_test_tfidf)

    # Tính toán các chỉ số
    acc = accuracy_score(y_test, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary")

    # LinearSVC dùng decision_function cho AUC
    scores = svm.decision_function(X_test_tfidf)
    auc = roc_auc_score(y_test, scores)

    results_storage[split] = acc
    print(f"✅ {split}: Acc={acc*100:.2f}% | F1={f1*100:.2f}% | AUC={auc:.4f}")

# 7. Tổng kết theo yêu cầu của bạn
in_domain_acc = results_storage["test1_nyt_mj"]
# Trung bình cộng 4 tập out-domain (test2, test3, test4, test5)
ood_acc_list = [results_storage[s] for s in test_splits[1:]]
avg_ood_acc = np.mean(ood_acc_list)

print(f"\n{'='*40}")
print(f"📊 KẾT QUẢ CUỐI CÙNG (TF-IDF + SVM)")
print(f"{'='*40}")
print(f"📍 In-domain (test1_nyt_mj) : {in_domain_acc*100:.2f}%")
print(f"🌐 Out-domain (Avg test2-5) : {avg_ood_acc*100:.2f}%")
print(f"{'='*40}")

# 8. Lưu mô hình
joblib.dump(tfidf, f"{OUTPUT_DIR}/tfidf_model.pkl")
joblib.dump(svm, f"{OUTPUT_DIR}/svm_model.pkl")
print(f"💾 Model saved to: {OUTPUT_DIR}")

Mounted at /content/drive
⏳ Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/655M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

...

Generating test2_bbc_dalle split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test3_cnn_dalle split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test4_bbc_sdxl split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test5_cnn_sdxl split:   0%|          | 0/500 [00:00<?, ? examples/s]

📊 Vectorizing text with TF-IDF...
🚀 Training Linear SVM...

🔍 ĐÁNH GIÁ TRÊN CÁC TẬP TEST:
✅ test1_nyt_mj: Acc=86.20% | F1=86.23% | AUC=0.9233
✅ test2_bbc_dalle: Acc=68.00% | F1=73.51% | AUC=0.8112
✅ test3_cnn_dalle: Acc=70.20% | F1=73.72% | AUC=0.7962
✅ test4_bbc_sdxl: Acc=68.80% | F1=74.17% | AUC=0.8189
✅ test5_cnn_sdxl: Acc=72.40% | F1=75.62% | AUC=0.8085

📊 KẾT QUẢ CUỐI CÙNG (TF-IDF + SVM)
📍 In-domain (test1_nyt_mj) : 86.20%
🌐 Out-domain (Avg test2-5) : 69.85%
💾 Model saved to: /content/drive/MyDrive/MFND_Baseline_TFIDF_SVM
